In [1]:
import os
import sys
import numpy as np
import warnings

sys.path.append('..')
import module

# Suppress warnings (supress_warning only runs under __main__)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from src.model.modules import AnxietyInferencer

In [2]:
CKPT_LOPO = './checkpoints_lopo/lopo_fold00_uar1.0000.pt'       # Best LOPO Checkpoint
CKPT_GKF  = './checkpoints/best_strategyC_fold04_uar0.9444.pt'  # Best GroupKFold Checkpoint

NORMALIZER_LOPO = './checkpoints_lopo/lopo_fold00_uar1.0000_normalizer.npz'
NORMALIZER_GKF  = './checkpoints/best_strategyC_fold04_uar0.9444_normalizer.npz'

infer_lopo = AnxietyInferencer(checkpoint_path = CKPT_LOPO,
                               normalizer_path = NORMALIZER_LOPO)

infer_gkf = AnxietyInferencer(checkpoint_path = CKPT_GKF,
                              normalizer_path = NORMALIZER_GKF)

Prediksi dari hasil cache numpy file yang berisikan fitur-fitur dari tiap clip video

In [3]:
import os
from pathlib import Path


BASE_DATASET_DIR = Path('/home/inadio/datasets/anxiety_raw')

NPY_PATH = os.path.join(BASE_DATASET_DIR, '.cache_npy', 'before', 'anxiety',
                        'm__firmansyah_1765171933077', 'q3',
                        'answer_3_ce8ef7e8-63e6-46c1-a510-1b8269c27ce7_sec.npy')

# ── Prediksi dengan kedua model ──────────────────────────────────────────
print('Prediksi dengan model LOPO:')
result_lopo = infer_lopo.predict_npy(NPY_PATH)
print(infer_lopo.explain(result_lopo))

print()
print('Prediksi dengan model GroupKFold:')
result_gkf  = infer_gkf.predict_npy(NPY_PATH)
print(infer_gkf.explain(result_gkf))

# ── Perbandingan singkat ──────────────────────────────────────────────────
print()
print('Perbandingan:')
print(f'  LOPO : {result_lopo.label.upper():4s}  '
      f'prob_high={result_lopo.prob_high:.4f}  '
      f'prob_low={result_lopo.prob_low:.4f}  '
      f'confidence={result_lopo.confidence:.1%}')
print(f'  GKF  : {result_gkf.label.upper():4s}  '
      f'prob_high={result_gkf.prob_high:.4f}  '
      f'prob_low={result_gkf.prob_low:.4f}  '
      f'confidence={result_gkf.confidence:.1%}')

if result_lopo.label == result_gkf.label:
    print(f'\n  Kedua model SEPAKAT: {result_lopo.label.upper()}')
else:
    print(f'\n  Kedua model TIDAK SEPAKAT — gunakan LOPO sebagai referensi utama')

Prediksi dengan model LOPO:
ANXIETY LEVEL PREDICTION RESULT
  Label     : HIGH
  Confidence: 81.8%
  Prob HIGH : 0.8180
  Prob LOW  : 0.1820
  Apex det. : 31

Top 10 Contributing Features:
  Feature                                Value  Direction
  -------------------------------------------------------
  apex2_eye_LR_d_mag                    0.6775  → LOW 
  apex1_right_eyebrow_net_dx           -0.3318  → HIGH
  apex1_eyebrow_LR_d_dx                 0.7826  → HIGH
  apex2_right_eyebrow_net_dx           -0.4481  → HIGH
  apex2_lips_net_dy                     0.2275  → HIGH
  apex2_eye_LR_d_dy                    -0.2471  → HIGH
  apex2_right_eye_mean_mag              0.0774  → HIGH
  apex3_lips_net_dx                     0.2489  → HIGH
  apex1_left_eyebrow_net_dx             0.4508  → HIGH
  apex3_left_eyebrow_net_dy             0.2265  → LOW 

Prediksi dengan model GroupKFold:
ANXIETY LEVEL PREDICTION RESULT
  Label     : LOW
  Confidence: 76.9%
  Prob HIGH : 0.2310
  Prob LOW  : 0.769

Prediksi dari video mentah dan belum pernah dikenali pada proses training

In [ ]:
from src.apex.modules.v2 import ApexPhaseSpotter


VIDEO_PATH = os.path.join(Path.home(), "skripkir", "pulse-live", ".assets", "20250520125029-anxious-val.avi")

result = infer_lopo.predict_vxideo(VIDEO_PATH)
print(infer_lopo.explain(result))

W0000 00:00:1774163916.875795    7723 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774163916.888048    7730 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774163916.900321    7730 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


ANXIETY LEVEL PREDICTION RESULT
  Label     : LOW
  Confidence: 66.8%
  Prob HIGH : 0.3317
  Prob LOW  : 0.6683
  Apex det. : 155

Top 10 Contributing Features:
  Feature                                Value  Direction
  -------------------------------------------------------
  apex3_right_eyebrow_net_dy            0.2003  → LOW 
  apex2_left_eyebrow_mean_mag           0.0328  → HIGH
  apex1_eyebrow_LR_d_mag               -0.3297  → LOW 
  apex3_left_eye_net_dy                -0.3518  → HIGH
  apex2_right_eye_net_dy                0.2167  → HIGH
  apex1_right_eye_net_dy                0.2098  → HIGH
  apex1_left_eye_net_dx                 0.2531  → HIGH
  apex1_eye_LR_d_dy                    -0.1899  → LOW 
  apex1_right_eyebrow_net_dy           -0.3206  → HIGH
  apex2_eye_LR_d_dx                    -0.2492  → LOW 


Prediksi video secara batching untuk beberapa video sekaligus

In [5]:
import glob
import pandas as pd

from pathlib import Path
from IPython.display import display


CACHE_DIR = Path('/home/inadio/datasets/anxiety_raw/.cache_npy/before/')

npy_files = sorted(glob.glob(str(CACHE_DIR / '**/*.npy'), recursive=True))

if not npy_files:
    print(f'Tidak ada file .npy ditemukan di: {CACHE_DIR}')
else:
    print(f'Ditemukan {len(npy_files)} file .npy')
    print(f'Contoh: {npy_files[0]}')
    print()

    # Jalankan batch prediction (gunakan subset dulu untuk test)
    sample_files = npy_files[:5]  # test dengan 5 file pertama
    print(f'Batch prediksi {len(sample_files)} clip:')
    results = infer_lopo.predict_batch(sample_files, input_type='npy')

    df_results = pd.DataFrame([{
        'file'      : os.path.basename(os.path.dirname(os.path.dirname(sample_files[i]))),
        'clip'      : os.path.basename(os.path.dirname(sample_files[i])),
        'label'     : r.label,
        'prob_high' : round(r.prob_high, 4),
        'prob_low'  : round(r.prob_low, 4),
        'confidence': round(r.confidence, 4),
        'n_apex'    : r.n_apex_detected,
        'warning'   : r.warning or '',
    } for i, r in enumerate(results.succeeded.values())])

    display(df_results)

Ditemukan 280 file .npy
Contoh: /home/inadio/datasets/anxiety_raw/.cache_npy/before/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765168488512/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec.npy

Batch prediksi 5 clip:


,file,clip,label,prob_high,prob_low,confidence,n_apex,warning
0,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q1,high,0.8072,0.1928,0.8072,38,
1,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q2,high,0.7906,0.2094,0.7906,33,
2,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q3,low,0.3057,0.6943,0.6943,27,
3,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q4,high,0.6955,0.3045,0.6955,42,
4,aaisyah_nursalsabiil_ni_patriarti_1765168488512,q5,high,0.5901,0.4099,0.5901,31,
